In [2]:
import os, time, sqlite3, requests, numpy as np, spotipy
from spotipy.oauth2 import SpotifyOAuth
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import pandas as pd
import json


In [3]:
#current problems. past 2024. rock past 2022. nu meetal/hard rock 2000s. last.fm .get similar. 


### Setup

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads variables from a .env file sitting next to this notebook

# ─── CONFIG ───────────────────────────────────────────────────────────────────
DB_PATH        = "songs.db"
LASTFM_API_KEY = os.environ.get("LASTFM_API_KEY")
LASTFM_URL     = "https://ws.audioscrobbler.com/2.0/"
RECCO_URL      = "https://api.reccobeats.com/v1"
MAX_RECCO_CALLS = 50
CLIENT_ID      = os.environ.get("SPOTIFY_CLIENT_ID")
CLIENT_SECRET  = os.environ.get("SPOTIFY_CLIENT_SECRET")
REDIRECT_URI   = "http://127.0.0.1:8080/callback"

assert LASTFM_API_KEY and CLIENT_ID and CLIENT_SECRET, "Missing credentials — check your .env file"

In [5]:
# Audio feature columns — mirrors original Spotify schema.
# key, mode, time_signature are omitted: ReccoBeats does not return them.
# They can be added back later by layering in AcousticBrainz for pre-2022 tracks.
FEATURE_COLS = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness",
    "valence", "tempo", "key", "mode"
]

# ─── SPOTIFY AUTH ─────────────────────────────────────────────────────────────
#if os.path.exists(".cache"):
#    os.remove(".cache")

sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id     = CLIENT_ID,
    client_secret = CLIENT_SECRET,
    redirect_uri  = REDIRECT_URI,
    scope         = "playlist-modify-public playlist-modify-private",
    open_browser  = True
))

### Note
- This will part will only work if you have ollama installed on your machine.
  You can still do the demo if you don't.
- Just make the path the database that was zipped.

In [6]:
# ── Ollama AI Tagger ──────────────────────────────────────────────────────────


OLLAMA_URL = "http://localhost:11434/api/generate" #
OLLAMA_MODEL = "llama3.1"  # run: ollama pull llama3.1
BATCH_SIZE = 10  # tracks per Ollama call

AI_TAG_COLS = [
    "has_autotune",
    "is_drum_heavy",
    "is_guitar_heavy",
    "dominant_instrument",
    "mood",
    "is_love_song",
    "is_explicit",
    "content_themes",
    "singer_gender",
    "singer_nationality",
    "band_nationality",
    "time_period",
    "period_genre_relevancy",
    "acoustic_or_electric",
    "has_distortion",
    "has_climax",
]

AI_TAG_COL_TYPES = {
    "has_autotune": "INTEGER",
    "is_drum_heavy": "INTEGER",
    "is_guitar_heavy": "INTEGER",
    "dominant_instrument": "TEXT",
    "mood": "TEXT",
    "is_love_song": "INTEGER",
    "is_explicit": "INTEGER",
    "content_themes": "TEXT",
    "singer_gender": "TEXT",
    "singer_nationality": "TEXT",
    "band_nationality": "TEXT",
    "time_period": "TEXT",
    "period_genre_relevancy": "TEXT",
    "acoustic_or_electric": "TEXT",
    "has_distortion": "INTEGER",
    "has_climax": "INTEGER",
}

### Main functions

In [7]:
#------acoustic brainz-----------------------------------------------
#
def get_acousticbrainz_features(name: str, artist: str) -> dict | None:
    """
    Resolves name + artist → MBID via MusicBrainz, then fetches
    low-level audio features from AcousticBrainz.
    Returns a dict keyed to FEATURE_COLS, or None if either step fails.
    """
    # ── Step 1: MusicBrainz MBID lookup ──────────────────────────────
    try:
        mb_url = "https://musicbrainz.org/ws/2/recording"
        params = {
            "query": f'recording:"{name}" AND artist:"{artist}"',
            "fmt": "json",
            "limit": 1
        }
        headers = {"User-Agent": "ReccoBeats/1.0 (your@email.com)"}
        r = requests.get(mb_url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        recordings = r.json().get("recordings", [])
        if not recordings:
            print(f"    ❌ MusicBrainz: no match for {name} — {artist}")
            return None
        mbid = recordings[0]["id"]
        print(f"    ✅ MusicBrainz MBID: {mbid}")
        time.sleep(1.1)  # MusicBrainz rate limit: 1 req/sec
    except Exception as e:
        print(f"    ❌ MusicBrainz error: {e}")
        return None

    # ── Step 2: AcousticBrainz feature fetch ─────────────────────────
    try:
        ab_url = f"https://acousticbrainz.org/{mbid}/low-level"
        r = requests.get(ab_url, timeout=10)
        if r.status_code == 404:
            print(f"    ❌ AcousticBrainz: no data for MBID {mbid}")
            return None
        r.raise_for_status()
        ab = r.json()

        rhythm   = ab.get("rhythm", {})
        tonal    = ab.get("tonal", {})
        lowlevel = ab.get("lowlevel", {})
        KEY_MAP  = {"C":0,"C#":1,"Db":1,"D":2,"D#":3,"Eb":3,"E":4,"F":5,
            "F#":6,"Gb":6,"G":7,"G#":8,"Ab":8,"A":9,"A#":10,"Bb":10,"B":11}
        MODE_MAP = {"major": 1, "minor": 0}
        


        features = {
            "danceability":     rhythm.get("danceability"),
            "energy":           lowlevel.get("average_loudness"),
            "loudness": (
                lowlevel.get("loudness_ebu128", {}).get("integrated")
                or lowlevel.get("average_loudness")
                or lowlevel.get("loudness", {}).get("mean")
            ),
            "speechiness":      lowlevel.get("mfcc", {}).get("mean", [None])[1],
            "acousticness":     lowlevel.get("hfc", {}).get("mean"),
            "instrumentalness": lowlevel.get("dissonance", {}).get("mean"),
            "liveness":         lowlevel.get("spectral_energy", {}).get("mean"),
            "valence":          tonal.get("chords_changes_rate"),
            "tempo":            rhythm.get("bpm"),
            "key":  KEY_MAP.get(tonal.get("key_key"), None),
            "mode": MODE_MAP.get(tonal.get("key_scale"), None),
        }

        missing = [k for k, v in features.items() if v is None]
        if missing:
            print(f"    ⚠️  AcousticBrainz: missing fields {missing}")

        return features

    except Exception as e:
        print(f"    ❌ AcousticBrainz error: {e}")
        return None
        
# ── DB Migration ──────────────────────────────────────────────────────────────

def migrate_ai_tag_cols():
    """Add AI tag columns to songs table if not already present."""
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute("PRAGMA table_info(songs)")
    existing = {row[1] for row in cur.fetchall()}
    for col, col_type in AI_TAG_COL_TYPES.items():
        if col not in existing:
            cur.execute(f"ALTER TABLE songs ADD COLUMN {col} {col_type}")
            print(f"  🔧 Migrated DB: added column '{col}'")
    con.commit()
    con.close()


# ── Cache Check ───────────────────────────────────────────────────────────────

def get_untagged_ids(spotify_ids: list) -> list:
    """Return IDs that haven't been AI-tagged yet."""
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    placeholders = ",".join(["?"] * len(spotify_ids))
    cur.execute(
        f"""SELECT spotify_track_id FROM songs
            WHERE spotify_track_id IN ({placeholders})
            AND mood IS NULL""",
        spotify_ids
    )
    untagged = {row[0] for row in cur.fetchall()}
    con.close()
    already_done = len(spotify_ids) - len(untagged)
    print(f"  ♻️  {already_done} already AI-tagged, {len(untagged)} to process.")
    return [sid for sid in spotify_ids if sid in untagged]


# ── Prompt Builder ────────────────────────────────────────────────────────────

def build_prompt(batch: list) -> str:
    """
    Build a prompt for a batch of tracks.
    Each item in batch is a dict: {spotify_track_id, title, artist, genre_tags}
    """
    tracks_text = ""
    for i, t in enumerate(batch):
        tracks_text += (
            f"{i+1}. Title: {t['title']} | "
            f"Artist: {t['artist']} | "
            f"Known genres: {t.get('genre_tags', 'unknown')}\n"
        )

    return f"""You are a music metadata expert. For each track below, return a JSON array where each element contains these exact keys:

- has_autotune: 1 or 0
- is_drum_heavy: 1 or 0
- is_guitar_heavy: 1 or 0
- dominant_instrument: string (e.g. "guitar", "piano", "synthesizer", "drums", "violin")
- mood: single word or short phrase (e.g. "euphoric", "melancholic", "aggressive", "calm")
- is_love_song: 1 or 0
- is_explicit: 1 or 0
- content_themes: comma-separated themes (e.g. "heartbreak,longing,loss")
- singer_gender: "male", "female", "mixed", or "unknown"
- singer_nationality: lowercase country (e.g. "american", "british", "canadian")
- band_nationality: lowercase country or "unknown"
- time_period: decade string (e.g. "1990s", "2000s", "2010s")
- period_genre_relevancy: one of "peak", "early", "late", "revival"
- acoustic_or_electric: "acoustic", "electric", or "mixed"
- has_distortion: 1 or 0
- has_climax: 1 or 0

Rules:
- Return ONLY a valid JSON array, no explanation, no markdown, no backticks.
- One object per track, in the same order as the input.
- If you are unsure about something, make your best guess. Never return null.
- period_genre_relevancy means how this song relates to its genre's timeline. 
  "peak" = released during the genre's most popular era,
  "early" = before the genre peaked,
  "late" = after the genre peaked,
  "revival" = much later, intentionally reviving the style.

Tracks:
{tracks_text}

Return only the JSON array:"""


# ── Ollama Call ───────────────────────────────────────────────────────────────

def call_ollama(prompt: str, expected: int) -> list:
    """Send prompt to Ollama and parse JSON array response."""
    try:
        resp = requests.post(OLLAMA_URL, json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.1,  # low temp = consistent structured output
                "num_predict": 2500,
            }
        }, timeout=120)
        resp.raise_for_status()
        raw = resp.json().get("response", "").strip()

        # Strip markdown fences if model adds them anyway
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        raw = raw.strip()

        parsed = json.loads(raw)
        if isinstance(parsed, list):
            return parsed[:expected]  # never return more than we asked for
        else:
            print("  ⚠️  Ollama returned non-list JSON, skipping batch.")
            return []

    except json.JSONDecodeError as e:
        print(f"  ⚠️  JSON parse error from Ollama: {e}")
        return []
    except Exception as e:
        print(f"  ⚠️  Ollama call failed: {e}")
        return []


# ── Save Tags ─────────────────────────────────────────────────────────────────

def save_ai_tags(spotify_id: str, tags: dict):
    """Write AI tag dict to the songs table."""
    # Only save keys we know about
    filtered = {k: tags[k] for k in AI_TAG_COLS if k in tags}
    if not filtered:
        return
    cols = ", ".join([f"{k} = ?" for k in filtered])
    vals = list(filtered.values()) + [spotify_id]
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute(
        f"UPDATE songs SET {cols} WHERE spotify_track_id = ?",
        vals
    )
    con.commit()
    con.close()


# ── Main Ingest ───────────────────────────────────────────────────────────────

def ingest_ai_tags(tracks: list):
    """
    Main entry point for Ollama AI tagging.

    `tracks` is a list of dicts with keys:
        - spotify_track_id
        - title
        - artist
        - genre_tags  (optional, from Last.fm — helps Ollama accuracy)
    """
    migrate_ai_tag_cols()

    spotify_ids = [t["spotify_track_id"] for t in tracks]
    to_tag = set(get_untagged_ids(spotify_ids))

    if not to_tag:
        print("  ✅ All tracks already AI-tagged.")
        return

    pending = [t for t in tracks if t["spotify_track_id"] in to_tag]
    batches = [pending[i:i+BATCH_SIZE] for i in range(0, len(pending), BATCH_SIZE)]

    print(f"  🤖 AI tagging {len(pending)} tracks in {len(batches)} batches...")

    total_tagged = 0
    for batch_num, batch in enumerate(batches, 1):
        print(f"  📦 Batch {batch_num}/{len(batches)} ({len(batch)} tracks)...")

        prompt = build_prompt(batch)
        results = call_ollama(prompt, expected=len(batch))

        if len(results) != len(batch):
            print(f"  ⚠️  Expected {len(batch)} results, got {len(results)}. Skipping batch.")
            continue

        for track, tags in zip(batch, results):
            save_ai_tags(track["spotify_track_id"], tags)
            total_tagged += 1
            print(f"  ✅ Tagged: {track['title']} by {track['artist']}")

        # Small pause between batches to keep Ollama happy
        if batch_num < len(batches):
            time.sleep(1)

    print(f"\n  🎉 AI tagging complete. {total_tagged}/{len(pending)} tracks tagged.")

# ─── DB SETUP ─────────────────────────────────────────────────────────────────
def migrate_db():
    """Add any missing columns to an existing songs table (safe to run every time)."""
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()

    # Get existing column names
    cur.execute("PRAGMA table_info(songs)")
    existing_cols = {row[1] for row in cur.fetchall()}

    # Add any FEATURE_COLS that are missing
    for col in FEATURE_COLS:
        if col not in existing_cols:
            cur.execute(f"ALTER TABLE songs ADD COLUMN {col} REAL")
            print(f"  🔧 Migrated DB: added column '{col}'")
    # In migrate_db(), after the FEATURE_COLS loop:
    cur.execute("PRAGMA table_info(songs)")
    existing_cols = {row[1] for row in cur.fetchall()}
    if "genre_tags" not in existing_cols:
        cur.execute("ALTER TABLE songs ADD COLUMN genre_tags TEXT")
        print("  🔧 Migrated DB: added column 'genre_tags'")

    con.commit()
    con.close()

def init_db():
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS songs (
            spotify_track_id TEXT PRIMARY KEY,
            name             TEXT,
            artists          TEXT,
            {", ".join([f"{c} REAL" for c in FEATURE_COLS])}
        )
    """)
    con.commit()
    con.close()
    migrate_db()  # ← always run after create, catches schema drift
def get_uncached_ids(spotify_ids: list) -> list:
    """Filter out IDs already in the DB — no need to re-fetch their features."""
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    placeholders = ",".join(["?"] * len(spotify_ids))
    cur.execute(
        f"SELECT spotify_track_id FROM songs WHERE spotify_track_id IN ({placeholders})",
        spotify_ids
    )
    cached = {row[0] for row in cur.fetchall()}
    con.close()
    new_ids = [sid for sid in spotify_ids if sid not in cached]
    print(f"  ♻️  {len(cached)} already cached, {len(new_ids)} new to fetch.")
    return new_ids
# ─── LAST.FM HELPERS ──────────────────────────────────────────────────────────
import re

def clean_title_for_lastfm(title: str) -> str:
    # Remove remaster notes, version tags, featured artists etc.
    title = re.sub(r'\s*[-–]\s*(Remaster(ed)?|Single Version|Live|Deluxe|Radio Edit).*', '', title, flags=re.IGNORECASE)
    title = re.sub(r'\s*\(feat\..*?\)', '', title, flags=re.IGNORECASE)
    title = re.sub(r'\s*\(.*?(Remaster|Version|Edit).*?\)', '', title, flags=re.IGNORECASE)
    return title.strip()
    
def lastfm_get(params):
    params.update({"api_key": LASTFM_API_KEY, "format": "json"})
    return requests.get(LASTFM_URL, params=params, timeout=5).json()

def get_similar_tracks(artist, track, limit=50):
    data = lastfm_get({
        "method":      "track.getSimilar",
        "artist":      artist,
        "track":       track,
        "limit":       limit,
        "autocorrect": 1
    })
    return data.get("similartracks", {}).get("track", [])
# ── Genre Tags (Last.fm) ────────────────────────────────────────────────────

def get_uncached_genre_ids(spotify_ids: list) -> list:
    """Return IDs missing genre_tags from the DB."""
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    placeholders = ",".join(["?"] * len(spotify_ids))
    cur.execute(
        f"""SELECT spotify_track_id FROM songs
            WHERE spotify_track_id IN ({placeholders})
            AND (genre_tags IS NULL OR genre_tags = '')""",
        spotify_ids
    )
    missing = {row[0] for row in cur.fetchall()}
    con.close()
    return [sid for sid in spotify_ids if sid in missing]


import requests


def fetch_lastfm_tags(artist: str, track: str) -> list:
    try:
        cleaned = clean_title_for_lastfm(track)
        base_params = {
            "api_key": LASTFM_API_KEY,
            "format": "json",
            "autocorrect": 1
        }

        # --- Tier 1: track.getTopTags ---
        resp = requests.get(LASTFM_URL, params={
            **base_params,
            "method": "track.getTopTags",
            "artist": artist,
            "track": cleaned,
        }, timeout=10)
        resp.raise_for_status()
        tags = resp.json().get("toptags", {}).get("tag", [])
        if isinstance(tags, dict):  # single-tag quirk
            tags = [tags]

        # --- Tier 2: track.getInfo ---
        if not tags:
            resp = requests.get(LASTFM_URL, params={
                **base_params,
                "method": "track.getInfo",
                "artist": artist,
                "track": cleaned,
            }, timeout=10)
            resp.raise_for_status()
            tags = resp.json().get("track", {}).get("toptags", {}).get("tag", [])
            if isinstance(tags, dict):  # single-tag quirk
                tags = [tags]

        # --- Tier 3: artist.getTopTags ---
        if not tags:
            resp = requests.get(LASTFM_URL, params={
                **base_params,
                "method": "artist.getTopTags",
                "artist": artist,
            }, timeout=10)
            resp.raise_for_status()
            tags = resp.json().get("toptags", {}).get("tag", [])
            if isinstance(tags, dict):  # single-tag quirk
                tags = [tags]

        NOISE = {"seen live", "favourites", "favorite", "love", "awesome",
                 "beautiful", "amazing", "cool", "good", "great", "best"}
        filtered = [
            t["name"].lower().strip()
            for t in tags
            if t["name"].lower().strip() not in NOISE
        ]

        return filtered[:5]

    except Exception as e:
        print(f"  ⚠️  Last.fm tag fetch failed for '{track}' by '{artist}': {e}")
        return []

def save_genre_tags(spotify_id: str, tags: list):
    """Store comma-separated genre tags for a track."""
    tag_str = ",".join(tags)
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute(
        "UPDATE songs SET genre_tags = ? WHERE spotify_track_id = ?",
        (tag_str, spotify_id)
    )
    con.commit()
    con.close()


def ingest_genre_tags(tracks: list):
    """
    Main entry point for genre tag ingestion.
    
    `tracks` is a list of dicts with keys:
        - spotify_track_id
        - artist  (string, primary artist name)
        - title   (string, track name)
    """
    spotify_ids = [t["spotify_track_id"] for t in tracks]
    to_tag = set(get_uncached_genre_ids(spotify_ids))

    if not to_tag:
        print("  ♻️  All tracks already have genre tags.")
        return

    print(f"  🏷️  Fetching genre tags for {len(to_tag)} tracks...")

    for track in tracks:
        sid = track["spotify_track_id"]
        if sid not in to_tag:
            continue

        tags = fetch_lastfm_tags(track["artist"], track["title"])

        if tags:
            save_genre_tags(sid, tags)
            print(f"  ✅ {track['title']} → {tags}")
        else:
            print(f"  ⚠️  No tags found for {track['title']}, skipping.")

        time.sleep(0.25)  # Last.fm rate limit buffer — 4 req/sec max
# ─── SPOTIFY SEARCH — name+artist → track ID + URI ────────────────────────────
def find_spotify_track(track_name: str, artist_name: str):
    # Check DB first before hitting Spotify
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute(
        "SELECT spotify_track_id FROM songs WHERE name = ? AND artists = ?",
        (track_name, artist_name)
    )
    row = cur.fetchone()
    con.close()
    time.sleep(1.0)
    if row:
        sid = row[0]
        return sid, f"spotify:track:{sid}"
    
    # Only hits Spotify if not cached
    query = f"track:{track_name} artist:{artist_name}"
    try:
        results = sp.search(q=query, type="track", limit=1)
        time.sleep(0.5)
        items = results["tracks"]["items"]
        if items:
            return items[0]["id"], items[0]["uri"]
    except Exception as e:
        print(f"  Spotify search failed for '{track_name}': {e}")
    
    return None, None

# ─── RECCOBEATS AUDIO FEATURES ────────────────────────────────────────────────
def get_recco_features_batch(spotify_ids: list) -> dict:
    if not spotify_ids:
        return {}
    try:
        resp = requests.get(
            f"{RECCO_URL}/audio-features",
            params={"ids": ",".join(spotify_ids)},
            timeout=10
        )
        
        # ── Hard stop on server errors — no point retrying ────────────
        if resp.status_code == 429:
            raise RuntimeError("🚨 ReccoBeats rate limit hit. Try again tomorrow.")
        if resp.status_code >= 500:
            raise RuntimeError(f"🚨 ReccoBeats server error ({resp.status_code}). Service may be down.")
        
        resp.raise_for_status()
        data = resp.json()

    except RuntimeError:
        raise  # let these bubble up to abort the batch loop
    except Exception as e:
        print(f"  ReccoBeats batch request failed: {e}")
        return {}

    result = {}
    for sid, item in zip(spotify_ids, data.get("content", [])):
        features = {col: item.get(col) for col in FEATURE_COLS}
        if all(v is not None for v in features.values()):
            result[sid] = features
    return result

# ─── INGESTION ────────────────────────────────────────────────────────────────
def ingest_tracks(track_list: list) -> list:
    init_db()
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()

    # ── Step 1: resolve Spotify IDs ──────────────────────────────────────────
    resolved = []
    for t in track_list:
        name   = t.get("name", "")
        artist = t.get("artist", "")
        if isinstance(artist, dict):
            artist = artist.get("name", "Unknown")

        track_id, uri = find_spotify_track(name, artist)
        if track_id:
            resolved.append((name, artist, track_id, uri))
        else:
            print(f"  ❌ Not on Spotify: {name} — {artist}")
        time.sleep(1.5)  # one sleep per track, not three

    if not resolved:
        print("No tracks resolved. Nothing to ingest.")
        con.close()
        return []

    # ── Step 2: batch-fetch ReccoBeats features ───────────────────────────────
    uncached_ids = get_uncached_ids([r[2] for r in resolved])
    all_features = {}

    if not uncached_ids:
        print("  ♻️  All tracks already cached. Skipping ReccoBeats entirely.")
    else:
        uncached_set = set(uncached_ids)
        BATCH_SIZE = 40
        id_list = [r[2] for r in resolved if r[2] in uncached_set]
        batches = [id_list[i:i+BATCH_SIZE] for i in range(0, len(id_list), BATCH_SIZE)]

        for batch_num, batch in enumerate(batches, 1):
            try:
                for attempt in range(2):
                    fetched = get_recco_features_batch(batch)
                    if fetched:
                        all_features.update(fetched)
                        break
                    else:
                        print(f"  ⚠️  Batch {batch_num} empty, retrying...")
                        time.sleep(2.0)
            except RuntimeError as e:
                print(e)
                print("  Aborting ReccoBeats fetching early.")
                break
            print(f"  ReccoBeats: fetched {len(all_features)} features so far...")
            time.sleep(0.5)

    # ── Step 3: write to DB ───────────────────────────────────────────────────
    # Pre-fetch all cached IDs in one query instead of one-by-one
    all_ids = [r[2] for r in resolved]
    placeholders = ",".join(["?"] * len(all_ids))
    cur.execute(f"SELECT spotify_track_id FROM songs WHERE spotify_track_id IN ({placeholders})", all_ids)
    already_in_db = {row[0] for row in cur.fetchall()}

    ingested = 0
    for name, artist, track_id, uri in resolved:
        features = all_features.get(track_id)

        if not features:
            if track_id in already_in_db:
                ingested += 1
                continue
            print(f"  🔄 Trying AcousticBrainz: {name} — {artist}")
            features = get_acousticbrainz_features(name, artist)
            if not features:
                print(f"  ⚠️  No features anywhere: {name} — {artist}")
                continue

        feat_vals = [features[c] for c in FEATURE_COLS]
        try:
            cur.execute(f"""
                INSERT INTO songs
                    (spotify_track_id, name, artists, {", ".join(FEATURE_COLS)})
                VALUES
                    (?, ?, ?, {", ".join(["?"] * len(FEATURE_COLS))})
                ON CONFLICT(spotify_track_id) DO NOTHING
            """, [track_id, name, artist] + feat_vals)
            ingested += 1
        except Exception as e:
            print(f"  ⚠️  DB error for {name}: {e}")

    con.commit()
    con.close()
    print(f"\n  Done! {ingested}/{len(track_list)} tracks stored in DB.")

    return [
        {"spotify_track_id": track_id, "title": name, "artist": artist}
        for name, artist, track_id, uri in resolved
    ]


def run(seed_artist: str, seed_track: str, k: int = 20):
    """
    Full end-to-end pipeline:
      1. Resolve seed track on Spotify
      2. Pull similar tracks from Last.fm
      3. Resolve all tracks on Spotify + batch-fetch audio features from ReccoBeats
      4. Fetch Last.fm genre tags
      5. AI-tag tracks with Ollama
      6. Run cosine similarity to rank recommendations
      7. Export TuneMyMusic .txt file
    """
    print("=" * 55)
    print("STEP 1 — Resolving seed track on Spotify...")
    print("=" * 55)
    seed_id, _ = find_spotify_track(seed_track, seed_artist)
    if not seed_id:
        print(f"❌ Could not find '{seed_track}' by '{seed_artist}' on Spotify. Aborting.")
        return
    print(f"   Seed Spotify ID: {seed_id}")

    print("\n" + "=" * 55)
    print("STEP 2 — Fetching similar tracks from Last.fm...")
    print("=" * 55)
    similar = get_similar_tracks(seed_artist, seed_track, limit=200)
    print(f"   Found {len(similar)} similar tracks.")
    all_tracks = [{"name": seed_track, "artist": seed_artist}] + [
        {"name": t["name"], "artist": t["artist"]} for t in similar
    ]

    print("\n" + "=" * 55)
    print("STEP 3 — Ingesting tracks (Spotify → ReccoBeats → DB)...")
    print("=" * 55)
    resolved_tracks = ingest_tracks(all_tracks)

    print("\n" + "=" * 55)
    print("STEP 4 — Fetching genre tags from Last.fm...")
    print("=" * 55)
    ingest_genre_tags(resolved_tracks)

    print("\n" + "=" * 55)
    print("STEP 5 — AI tagging with Ollama... sike i lied")
    print("=" * 55)
    #ingest_ai_tags(resolved_tracks)

    print("\n" + "=" * 55)
    print("STEP 6 — Running recommendation engine...")
    print("=" * 55)
    try:
        recs = recommend(seed_id, k=k)
    except ValueError as e:
        print(f"❌ {e}")
        return
    if not recs:
        print("No recommendations generated. Try ingesting more tracks first.")
        return
    for i, r in enumerate(recs):
        print(f"  {i+1:>2}. {r['name']} — {r['artists']}  (similarity: {r['similarity']})")

    print("\n" + "=" * 55)
    print("STEP 7 — Exporting TuneMyMusic .txt file...")
    print("=" * 55)
    export_tunemymusic_txt(recs, seed_name=seed_track)

# ─── RECOMMENDATION ENGINE ────────────────────────────────────────────────────
def load_matrix():
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    cur.execute(f"""
        SELECT spotify_track_id, name, artists, {', '.join(FEATURE_COLS)}
        FROM songs
    """)
    rows = cur.fetchall()
    con.close()
    ids  = [r[0] for r in rows]
    meta = [{"id": r[0], "name": r[1], "artists": r[2]} for r in rows]
    X    = np.array([r[3:] for r in rows], dtype=float)
    return ids, meta, X

def recommend(track_id: str, k: int = 20):
    """
    Returns top-k recommendations for a given Spotify track ID.
    The track must already be in the DB (run ingest_tracks first).
    """
    ids, meta, X = load_matrix()

    if track_id not in ids:
        raise ValueError(f"'{track_id}' not in DB. Run ingest_tracks first.")

    scaler = StandardScaler()
    Xs     = scaler.fit_transform(X)
    nn     = NearestNeighbors(metric="cosine", algorithm="brute")
    nn.fit(Xs)

    idx = ids.index(track_id)
    print(f'\n🎵 Seed: {meta[idx]["name"]} — {meta[idx]["artists"]}')

    dists, idxs = nn.kneighbors(
        Xs[idx].reshape(1, -1), n_neighbors=min(k + 1, len(ids))
    )

    recs = []
    for j, d in zip(idxs[0], dists[0]):
        if ids[j] == track_id:
            continue
        recs.append({**meta[j], "similarity": round(1 - d, 3)})
        if len(recs) >= k:
            break
    return recs# ── Filter Query Layer ────────────────────────────────────────────────────────

from typing import Any

def build_filter_query(filters: dict, logic: str = "AND") -> tuple[str, list]:
    """
    Build a SQL WHERE clause from a filter dict.

    Supported filter keys and their expected values:
    ┌─────────────────────────┬────────────────────────────────────────────────┐
    │ Key                     │ Value                                          │
    ├─────────────────────────┼────────────────────────────────────────────────┤
    │ genre_tags              │ list[str]  e.g. ["country", "outlaw"]          │
    │ mood                    │ str        e.g. "melancholic"                  │
    │ time_period             │ str        e.g. "1970s"                        │
    │ period_genre_relevancy  │ str        e.g. "peak"                         │
    │ singer_gender           │ str        e.g. "male"                         │
    │ singer_nationality      │ str        e.g. "american"                     │
    │ band_nationality        │ str        e.g. "american"                     │
    │ acoustic_or_electric    │ str        e.g. "acoustic"                     │
    │ dominant_instrument     │ str        e.g. "guitar"                       │
    │ content_themes          │ list[str]  e.g. ["heartbreak", "loss"]         │
    │ is_love_song            │ bool                                           │
    │ is_explicit             │ bool                                           │
    │ has_autotune            │ bool                                           │
    │ is_drum_heavy           │ bool                                           │
    │ is_guitar_heavy         │ bool                                           │
    │ has_distortion          │ bool                                           │
    │ has_climax              │ bool                                           │
    │ bpm_min                 │ float      minimum tempo                       │
    │ bpm_max                 │ float      maximum tempo                       │
    │ energy_min              │ float      0.0 – 1.0                           │
    │ energy_max              │ float      0.0 – 1.0                           │
    │ valence_min             │ float      0.0 – 1.0 (sad → happy)            │
    │ valence_max             │ float      0.0 – 1.0                           │
    │ acousticness_min        │ float      0.0 – 1.0                           │
    │ acousticness_max        │ float      0.0 – 1.0                           │
    │ danceability_min        │ float      0.0 – 1.0                           │
    │ danceability_max        │ float      0.0 – 1.0                           │
    │ instrumentalness_min    │ float      0.0 – 1.0                           │
    │ instrumentalness_max    │ float      0.0 – 1.0                           │
    └─────────────────────────┴────────────────────────────────────────────────┘

    logic: "AND" (all filters must match) or "OR" (any filter can match)

    Returns: (where_clause_string, params_list)
    """
    clauses = []
    params  = []

    # ── Boolean flags ─────────────────────────────────────────────────────────
    bool_fields = [
        "is_love_song", "is_explicit", "has_autotune",
        "is_drum_heavy", "is_guitar_heavy", "has_distortion", "has_climax"
    ]
    for field in bool_fields:
        if field in filters:
            clauses.append(f"{field} = ?")
            params.append(1 if filters[field] else 0)

    # ── Exact text matches ────────────────────────────────────────────────────
    text_fields = [
        "mood", "time_period", "period_genre_relevancy",
        "singer_gender", "singer_nationality", "band_nationality",
        "acoustic_or_electric", "dominant_instrument"
    ]
    for field in text_fields:
        if field in filters:
            clauses.append(f"LOWER({field}) = LOWER(?)")
            params.append(filters[field])

    # ── List fields (genre_tags, content_themes) — LIKE match per item ───────
    for field in ["genre_tags", "content_themes"]:
        if field in filters:
            values = filters[field]
            if isinstance(values, str):
                values = [values]
            sub_clauses = [f"{field} LIKE ?" for _ in values]
            # Items within a list field always AND together
            clauses.append(f"({' AND '.join(sub_clauses)})")
            params.extend([f"%{v}%" for v in values])

    # ── Numeric ranges ────────────────────────────────────────────────────────
    range_fields = {
        "bpm_min":              ("tempo",             ">="),
        "bpm_max":              ("tempo",             "<="),
        "energy_min":           ("energy",            ">="),
        "energy_max":           ("energy",            "<="),
        "valence_min":          ("valence",           ">="),
        "valence_max":          ("valence",           "<="),
        "acousticness_min":     ("acousticness",      ">="),
        "acousticness_max":     ("acousticness",      "<="),
        "danceability_min":     ("danceability",      ">="),
        "danceability_max":     ("danceability",      "<="),
        "instrumentalness_min": ("instrumentalness",  ">="),
        "instrumentalness_max": ("instrumentalness",  "<="),
        "key_min":              ("key",               ">="),   # ← add
        "key_max":              ("key",               "<="),   # ← add
    }
    for filter_key, (col, op) in range_fields.items():
        if filter_key in filters:
            clauses.append(f"{col} {op} ?")
            params.append(filters[filter_key])

    if not clauses:
        return "", []

    joiner = f" {logic.strip().upper()} "
    where  = joiner.join(clauses)
    return where, params


def query_tracks(filters: dict, logic: str = "AND") -> list[dict]:
    """
    Return all tracks from the DB matching the given filters.
    No similarity ranking — raw filter results only.
    """
    where, params = build_filter_query(filters, logic)

    sql = "SELECT * FROM songs"
    if where:
        sql += f" WHERE {where}"

    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute(sql, params)
    rows = [dict(row) for row in cur.fetchall()]
    con.close()

    print(f"  🔍 Filter query returned {len(rows)} tracks.")
    return rows


def recommend_filtered(
    seed_id:  str,
    filters:  dict,
    logic:    str = "AND",
    k:        int = 20
) -> list[dict]:
    """
    Filter first, then rank surviving tracks by cosine similarity to the seed.

    Args:
        seed_id:  Spotify track ID of the seed track
        filters:  dict of filter criteria (see build_filter_query for keys)
        logic:    "AND" or "OR" — how filters combine
        k:        number of recommendations to return

    Returns:
        list of dicts with name, artists, similarity, and all tag fields
    """
    import numpy as np
    from sklearn.neighbors import NearestNeighbors

    # ── Pull seed vector ──────────────────────────────────────────────────────
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    cur = con.cursor()
    cur.execute("SELECT * FROM songs WHERE spotify_track_id = ?", (seed_id,))
    seed_row = cur.fetchone()
    con.close()

    if not seed_row:
        raise ValueError(f"Seed track '{seed_id}' not found in DB.")

    seed_row = dict(seed_row)
    seed_vec = [seed_row.get(col) for col in FEATURE_COLS]
    if any(v is None for v in seed_vec):
        raise ValueError("Seed track is missing audio features.")

    # ── Pull filtered candidate pool ──────────────────────────────────────────
    candidates = query_tracks(filters, logic=logic)

    # Remove seed from candidates
    candidates = [c for c in candidates if c["spotify_track_id"] != seed_id]

    # Remove any candidates missing audio features
    def is_numeric(v):
        try:
            float(v)
            return True
        except (TypeError, ValueError):
            return False

    candidates = [
        c for c in candidates
        if all(is_numeric(c.get(col)) for col in FEATURE_COLS)
    ]

    if not candidates:
        print("  ⚠️  No candidates survived filtering. Try relaxing your filters.")
        return []

    if len(candidates) < k:
        print(f"  ⚠️  Only {len(candidates)} candidates after filtering (requested {k}).")
        k = len(candidates)

    # ── Cosine similarity ranking ─────────────────────────────────────────────
    from sklearn.preprocessing import StandardScaler

# ── Cosine similarity ranking ─────────────────────────────────────────────────
    matrix  = np.array([[c[col] for col in FEATURE_COLS] for c in candidates])
    seed_np = np.array(seed_vec).reshape(1, -1)

# Stack seed with candidates so they're scaled together
    combined = np.vstack([seed_np, matrix])
    scaler   = StandardScaler()
    combined_scaled = scaler.fit_transform(combined)

    seed_scaled       = combined_scaled[0].reshape(1, -1)
    candidates_scaled = combined_scaled[1:]


    nn = NearestNeighbors(n_neighbors=k, metric="cosine")
    nn.fit(candidates_scaled)
    distances, indices = nn.kneighbors(seed_scaled)


    results = []
    for dist, idx in zip(distances[0], indices[0]):
        c = candidates[idx]
        results.append({
            "name":                  c["name"],
            "artists":               c["artists"],
            "spotify_track_id":      c["spotify_track_id"],
            "similarity":            round(1 - dist, 4),
            "mood":                  c.get("mood"),
            "genre_tags":            c.get("genre_tags"),
            "time_period":           c.get("time_period"),
            "singer_gender":         c.get("singer_gender"),
            "acoustic_or_electric":  c.get("acoustic_or_electric"),
            "is_love_song":          c.get("is_love_song"),
            "is_explicit":           c.get("is_explicit"),
            "key":                   c.get("key"),   # ← add this
        })

    return results
# ─── TUNEMYMUSIC TXT EXPORT ───────────────────────────────────────────────────
def export_tunemymusic_txt(recs: list, seed_name: str) -> str:
    """
    Exports recommendations as a TuneMyMusic-compatible plain-text file.
    Format: one track per line → "Song Title - ARTIST NAME"
    """
    safe_seed = "".join(c for c in seed_name if c.isalnum() or c in " _-").strip()
    filename  = f"tunemymusic_{safe_seed.replace(' ', '_')}.txt"

    lines = [f"{rec['name']} - {rec['artists'].upper()}" for rec in recs]

    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

    print(f"\n📄 TuneMyMusic export saved → {filename}")
   
    return filename

# ─── FULL PIPELINE ────────────────────────────────────────────────────────────
def run(seed_artist: str, seed_track: str, k: int = 20, filters: dict = None, logic: str = "AND"):  
    """
    Full end-to-end pipeline:
      1. Resolve seed track on Spotify to get its track ID
      2. Pull similar tracks from Last.fm
      3. Resolve all tracks on Spotify + batch-fetch audio features from ReccoBeats
      4. Store everything in SQLite
      5. Run cosine similarity to rank recommendations
      6. Export TuneMyMusic .txt file

    Usage:
        run("The Weeknd", "Blinding Lights", k=20)
    """
    print("=" * 55)
    print(f"STEP 1 — Resolving seed track on Spotify...")
    print("=" * 55)
    seed_id, _ = find_spotify_track(seed_track, seed_artist)
    if not seed_id:
        print(f"❌ Could not find '{seed_track}' by '{seed_artist}' on Spotify. Aborting.")
        return
    print(f"   Seed Spotify ID: {seed_id}")

    print("\n" + "=" * 55)
    print("STEP 2 — Fetching similar tracks from Last.fm...")
    print("=" * 55)
    similar = get_similar_tracks(seed_artist, seed_track, limit=200)
    print(f"   Found {len(similar)} similar tracks.")

    all_tracks = [{"name": seed_track, "artist": seed_artist}] + [
        {"name": t["name"], "artist": t["artist"]} for t in similar
    ]

    print("\n" + "=" * 55)
    print("STEP 3 — Ingesting tracks (Spotify search → ReccoBeats → DB)...")
    print("=" * 55)
    resolved_tracks = ingest_tracks(all_tracks)
    ingest_genre_tags(resolved_tracks)
    ingest_ai_tags(resolved_tracks)
    print("\n" + "=" * 55)
    print("STEP 4 — Running recommendation engine...")
    print("=" * 55)
    try:
        recs = recommend_filtered(seed_id, filters=filters or {}, logic=logic, k=k)
    except ValueError as e:
        print(f"❌ {e}")
        return

    if not recs:
        print("No recommendations generated. Try ingesting more tracks first.")
        return

    for i, r in enumerate(recs):
        print(f"  {i+1:>2}. {r['name']} — {r['artists']}  (similarity: {r['similarity']})")

    print("\n" + "=" * 55)
    print("STEP 5 — Exporting TuneMyMusic .txt file...")
    print("=" * 55)
    export_tunemymusic_txt(recs, seed_name=seed_track)

# ─── ENTRY POINT ──────────────────────────────────────────────────────────────

# ─── DB & AUTH CLEANUP ────────────────────────────────────────────────────────

### Main

In [8]:
init_db()

Full Pipeline with AI tags

In [11]:
#results = run(
#    seed_artist="Linkin Park",
 #   seed_track="Somewhere I Belong",
  #  k=10,
   # filters={
    #    "acoustic_or_electric": "electric",
     #   "has_distortion": 1,
      #  "period_genre_relevancy": "peak",
       # "genre_tags": ["nu metal"]
#    },
 #   logic="AND"
#)

This can take a while. If you want to see the cosine similarity part, do the non run one below.

Without AI tags. Wouldn't recommend

In [12]:
#run(seed_artist="Beastie Boys",
#    seed_track="Sabotage",
#    k=20,
#    filters={}
#   )

STEP 1 — Resolving seed track on Spotify...
   Seed Spotify ID: 0Puj4YlTm6xNzDDADXHMI9

STEP 2 — Fetching similar tracks from Last.fm...
   Found 196 similar tracks.

STEP 3 — Ingesting tracks (Spotify search → ReccoBeats → DB)...


HTTP Error for GET to https://api.spotify.com/v1/search with Params: {'q': 'track:Intergalactic artist:Beastie Boys', 'limit': 1, 'offset': 0, 'type': 'track', 'market': None} returned 403 due to Active premium subscription required for the owner of the app. When the subscription status changes, it can take a few hours before requests are allowed again.


  Spotify search failed for 'Intergalactic': http status: 403, code: -1 - https://api.spotify.com/v1/search?q=track%3AIntergalactic+artist%3ABeastie+Boys&limit=1&offset=0&type=track:
 Active premium subscription required for the owner of the app. When the subscription status changes, it can take a few hours before requests are allowed again., reason: None
  ❌ Not on Spotify: Intergalactic — Beastie Boys


KeyboardInterrupt: 

This turns the whole database if you want to see what the math is being done on

In [10]:
con = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM songs", con)
con.close()

pd.set_option('display.max_columns', None)
df

,spotify_track_id,name,artists,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,key,mode,genre_tags,has_autotune,is_drum_heavy,is_guitar_heavy,dominant_instrument,mood,is_love_song,is_explicit,content_themes,singer_gender,singer_nationality,band_nationality,time_period,period_genre_relevancy,acoustic_or_electric,has_distortion,has_climax
0,20C8yXjwDzGkwkJ9FYxpR3,If We Make It Through December,Merle Haggard,0.484000,0.413000,-14.422000,0.039400,0.633000,0.044200,0.375000,0.766000,106.720000,2.0,1.0,"country,classic country,outlaw country,singer-...",0,0.0,1,guitar,melancholic,0,0,,male,american,unknown,1970s,early,electric,1,1
1,5DMGnZllsLYdfd5dvia1bz,Silver Wings,Merle Haggard,0.623000,0.397000,-14.483000,0.029200,0.319000,0.000000,0.260000,0.644000,88.822000,10.0,1.0,"country,classic country,outlaw country,singer-...",0,0.0,1,guitar,melancholic,0,0,,male,american,unknown,1970s,early,electric,1,1
2,6sApjp472JTWuNky6pnvJb,Big City,Merle Haggard,0.454000,0.297000,-10.015000,0.033600,0.978000,0.002290,0.098400,0.644000,179.855000,1.0,1.0,"country,classic country,outlaw country,singer-...",0,0.0,1,guitar,melancholic,0,0,,male,american,unknown,1970s,early,electric,1,1
3,4KmAGKJbeY2DUiLInlet53,Good Hearted Woman,Waylon Jennings,0.733000,0.441000,-12.548000,0.031500,0.631000,0.000667,0.115000,0.784000,124.547000,2.0,1.0,"country,outlaw country,classic country,waylon ...",0,0.0,1,guitar,melancholic,0,0,,male,american,unknown,1970s,early,electric,1,1
4,6ANPIv5r3vdWntrmFa6H6M,"Luckenbach, Texas (Back to the Basics of Love)...",Waylon Jennings,0.601000,0.766000,-13.262000,0.046600,0.586000,0.000903,0.347000,0.856000,130.140000,7.0,1.0,"country,outlaw country,classic country,waylon ...",0,0.0,1,guitar,melancholic,0,0,,male,american,unknown,1970s,early,electric,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1460,3yGy1JYz3zQKlxSgjgpQqX,Praise You,Fatboy Slim,0.707000,0.947000,-1.192000,0.275000,0.008780,0.000078,0.366000,0.738000,94.932000,1.0,1.0,"electronic,dance,big beat,90s,electronica",0,0.0,0,synthesizer,euphoric,0,1,,male,british,unknown,1990s,peak,electric,1,1
1461,4Vdgnoec2ZeJ4aNa4uFqIa,Fame,Vanilla Ice,0.676000,0.905000,-2.569000,0.147000,0.594000,0.000000,0.154000,0.519000,152.186000,1.0,1.0,"rap,hip-hop,90s,hip hop,rapcore",0,0.0,0,synthesizer,aggressive,0,1,,male,american,unknown,1990s,peak,electric,1,1
1462,3jlUMYwPBzVUBqfhcScVDX,What I'm After,Lords of the Underground,0.618000,0.716000,-8.224000,0.036700,0.004730,0.822000,0.179000,0.545000,92.174000,6.0,0.0,"hip-hop,underground hip-hop,rap,east coast rap...",0,0.0,0,synthesizer,aggressive,0,1,,male,american,unknown,1990s,peak,electric,1,1
1463,4ML7ozvpaVwpMi163FWk7e,Revolutionary Beat,Flipsyde,1.242318,0.893249,0.893249,99.213524,31.718235,0.475927,0.025468,0.083295,162.501511,5.0,1.0,"hip-hop,rap,rapcore,rock,hip hop",0,0.0,0,synthesizer,aggressive,0,1,,male,american,unknown,1990s,peak,electric,1,1


Faster but it does not write to a text file

In [10]:
conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.execute("SELECT spotify_track_id FROM songs WHERE name = ? AND artists LIKE ?", 
            ("Smells Like Teen Spirit", "%Nirvana%"))
row = cur.fetchone()

if row:
    seed_id = row[0]
    # Add key/mode to range_fields in build_filter_query, OR just pre-filter here:
    recs = recommend_filtered(seed_id, filters={}, k=1210) #add filters if wanted
    print(f"Total recs: {len(recs)}")



    
    for i, r in enumerate(recs):
        print(f"  {i+1:>2}. {r['name']} — {r['artists']}  (similarity: {r['similarity']})")
else:
    print("Song not found in DB")

conn.close()

  🔍 Filter query returned 1465 tracks.
Total recs: 1210
   1. Clueless — Beach Bunny  (similarity: 0.9544)
   2. Skin — Madonna  (similarity: 0.9462)
   3. This Love — Maroon 5  (similarity: 0.9459)
   4. Till The Wheels Fall Off (feat. Lil Durk & Capella Grey) — Chris Brown  (similarity: 0.9271)
   5. Social Cues — Cage the Elephant  (similarity: 0.9242)
   6. Little Dark Age — MGMT  (similarity: 0.9176)
   7. Sorry — Nothing But Thieves  (similarity: 0.9176)
   8. That’s So True — Gracie Abrams  (similarity: 0.9087)
   9. The Sharpest Lives — My Chemical Romance  (similarity: 0.9046)
  10. Perm — Bruno Mars  (similarity: 0.8995)
  11. Tip Toes — half•alive  (similarity: 0.8904)
  12. Strange - Single Version — Patsy Cline  (similarity: 0.887)
  13. Mmm Mmm Mmm Mmm — Crash Test Dummies  (similarity: 0.8865)
  14. Boulevard of Broken Dreams — Green Day  (similarity: 0.8791)
  15. I Don't Care — Cheryl Cole  (similarity: 0.8761)
  16. Sick of It — Skillet  (similarity: 0.8718)
  17. Vic